In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 16:42:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.option("recursiveFileLookup", "true").parquet("data/pq/green")

In [3]:
df_green.registerTempTable('green')

/home/abrar/spark-batch-job/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [12]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [10]:
df_green_revenue.show()

[Stage 7:>                                                          (0 + 4) / 4]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|   7| 769.7299999999996|            45|
|2020-01-01 00:00:00|  17|195.03000000000003|             9|
|2020-01-01 00:00:00|  18|               7.8|             1|
|2020-01-01 00:00:00|  22|              15.8|             1|
|2020-01-01 00:00:00|  24|              87.6|             3|
|2020-01-01 00:00:00|  25| 531.0000000000002|            26|
|2020-01-01 00:00:00|  29|              61.3|             1|
|2020-01-01 00:00:00|  32| 68.94999999999999|             2|
|2020-01-01 00:00:00|  33|317.27000000000004|            11|
|2020-01-01 00:00:00|  35|            129.96|             5|
|2020-01-01 00:00:00|  36|295.34000000000003|            11|
|2020-01-01 00:00:00|  37|            175.67|             6|
|2020-01-01 00:00:00|  38| 98.78999999999999|             2|
|2020-01-01 00:00:00|  4

In [8]:
spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    lpep_pickup_datetime
FROM
    green
LIMIT 20
""").show()

+-------------------+--------------------+
|               hour|lpep_pickup_datetime|
+-------------------+--------------------+
|2020-01-23 13:00:00| 2020-01-23 13:10:15|
|2020-01-20 15:00:00| 2020-01-20 15:09:00|
|2020-01-15 20:00:00| 2020-01-15 20:23:41|
|2020-01-05 16:00:00| 2020-01-05 16:32:26|
|2020-01-29 19:00:00| 2020-01-29 19:22:42|
|2020-01-15 11:00:00| 2020-01-15 11:07:42|
|2020-01-16 08:00:00| 2020-01-16 08:22:29|
|2020-01-28 17:00:00| 2020-01-28 17:05:28|
|2020-01-22 14:00:00| 2020-01-22 14:51:37|
|2020-01-31 10:00:00| 2020-01-31 10:25:04|
|2020-01-20 15:00:00| 2020-01-20 15:50:54|
|2020-01-31 11:00:00| 2020-01-31 11:35:17|
|2020-01-04 20:00:00| 2020-01-04 20:44:28|
|2020-01-17 21:00:00| 2020-01-17 21:47:52|
|2020-01-21 23:00:00| 2020-01-21 23:13:47|
|2020-01-02 08:00:00| 2020-01-02 08:11:21|
|2020-01-27 02:00:00| 2020-01-27 02:59:20|
|2020-01-16 14:00:00| 2020-01-16 14:39:13|
|2020-01-16 18:00:00| 2020-01-16 18:42:24|
|2020-01-03 09:00:00| 2020-01-03 09:24:54|
+----------

In [13]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [20]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

In [22]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [27]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [46]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [47]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [48]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [50]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [51]:
df_join = spark.read.parquet('data/report/revenue/total')

In [56]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [54]:
df_zones = spark.read.parquet('zones/')

In [57]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [62]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')